# Temat: Predykcja popytu na produkty

### Stakeholderzy: Dział sprzedaży, marketing, zarządzanie łańcuchem dostaw, klienci.
### Opis: Modelowanie popytu na produkty w różnych okresach, aby zoptymalizować zapasy i produkcję.

### Dataset: https://www.kaggle.com/c/rossmann-store-sales

### Authors:
- Joanna Sawicka
- Kiryl Radkevich
- Robert Walkiewicz
- Marek Lipiński

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


train = pd.read_csv('..//data/train.csv')
store = pd.read_csv('..//data/store.csv')

print(train.shape)
print(store.shape)

(1017209, 9)
(1115, 10)


C:\Users\marek.lipinski\AppData\Local\Temp\ipykernel_26660\526895874.py:7: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('..//data/train.csv')


In [27]:
train.head()


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [28]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [29]:
store.info()

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 98.0 KB


In [30]:
train.isnull().sum()

Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

In [31]:
store.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

In [32]:
df = pd.merge(train, store, on='Store', how='left')
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [33]:
df.isna().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Subtask DS24G1-25 - Imputation of missing data - ML

In [34]:
#Based on kaggle: 
# CompetitionDistance - distance in meters to the nearest competitor store




In [35]:
max_distance = df['CompetitionDistance'].max()   
max_distance 

75860.0

In [36]:
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(max_distance) # If null in row - store have no competitors in neighbourhood. Fill with max. value is the most reasonable choice.

In [37]:
# CompetitionOpenSince[Month/Year] - gives the approximate year and month of the time the nearest competitor was opened
df['CompetitionOpenSinceMonth']

0           9.0
1          11.0
2          12.0
3           9.0
4           4.0
           ... 
1017204     6.0
1017205     4.0
1017206     NaN
1017207     NaN
1017208     NaN
Name: CompetitionOpenSinceMonth, Length: 1017209, dtype: float64

In [38]:
df['CompetitionOpenSinceYear'].max()

2015.0

In [39]:
df[df['CompetitionOpenSinceYear'] == 2015]['CompetitionOpenSinceMonth'].max()

8.0

In [40]:
# filling with month 12 (Dec) allows later to push the missing dates into the future (negative value in result). Important in feature eng. !!
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(12)

In [41]:
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(2015)
df.isna().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance               0
CompetitionOpenSinceMonth         0
CompetitionOpenSinceYear          0
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

In [42]:
#Based on kaggle: Promo2Since[Year/Week] - describes the year and calendar week when the store started participating in Promo2

In [43]:
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0) # 0 means there was no promo
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(2015) # 2015 to get 0 when calculating promo period
df['PromoInterval'] = df['PromoInterval'].fillna("None") # "None" for stores without promo


In [44]:
df.isna().sum()

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64

In [45]:
# End of task - complete Promo2 imputation and clear all missing values

## Data Type Conversion

The purpose of this step is to convert columns to appropriate data types to improve data processing, enable time-series analysis, and prepare the dataset for machine learning models.

In [46]:
# Check data types
df.dtypes

Store                          int64
DayOfWeek                      int64
Date                             str
Sales                          int64
Customers                      int64
Open                           int64
Promo                          int64
StateHoliday                  object
SchoolHoliday                  int64
StoreType                        str
Assortment                       str
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                    str
dtype: object

In [47]:
# Convert Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Convert categorical variables to category type
categorical_columns = [
    'StoreType',
    'Assortment',
    'StateHoliday',
    'PromoInterval'
]

for col in categorical_columns:
    df[col] = df[col].astype('category')

# Verify data type changes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   DayOfWeek                  1017209 non-null  int64         
 2   Date                       1017209 non-null  datetime64[us]
 3   Sales                      1017209 non-null  int64         
 4   Customers                  1017209 non-null  int64         
 5   Open                       1017209 non-null  int64         
 6   Promo                      1017209 non-null  int64         
 7   StateHoliday               1017209 non-null  category      
 8   SchoolHoliday              1017209 non-null  int64         
 9   StoreType                  1017209 non-null  category      
 10  Assortment                 1017209 non-null  category      
 11  CompetitionDistance        1017209 non-null  flo

In [48]:
# Extract date-related features

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week

## Anomaly and Logical Error Analysis


In [49]:
negative_sales = df[df['Sales'] < 0].shape[0]
negative_customers = df[df['Customers'] < 0].shape[0]

closed_with_sales = df[(df['Open'] == 0) & (df['Sales'] > 0)].shape[0]
open_with_no_sales = df[(df['Open'] == 1) & (df['Sales'] == 0)].shape[0]

print(f"Number of records with negative sales: {negative_sales}")
print(f"Number of records with negative customers: {negative_customers}")
print(f"Number of records (Store closed, but Sales > 0): {closed_with_sales}")
print(f"Number of records (Store open, but Sales == 0): {open_with_no_sales}")

Number of records with negative sales: 0
Number of records with negative customers: 0
Number of records (Store closed, but Sales > 0): 0
Number of records (Store open, but Sales == 0): 54


In [50]:
df = df[df['Sales'] > 0]
df = df[df['Customers'] >= 0]

print(f"Dataset shape after filtering anomalies: {df.shape}")

Dataset shape after filtering anomalies: (844338, 22)
